In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
df_weg = pd.read_excel('Weg_completo.xlsx', sheet_name='WEG FACTURAS ND NC ANUAL')

# Limpiamos columnas vacías generadas por la exportación
df_weg = df_weg.loc[:, ~df_weg.columns.str.contains('^Unnamed')]

df_weg

,CPTE.,FECHA,CONCEPTO,DESCRIPCIÓN,CANT.,PRECIO/MONTO,IVA,PRECIO+IVA,TOTAL
0,FC-A-3-11242,2025-01-03,NaN,BLANCO POLAR (16466355),50,7367.500000,21.0,8914.6750,3.683750e+05
1,FC-A-3-11242,2025-01-03,NaN,ALUMINIO BARA XP 10-0040 (16363909),25,8420.000000,21.0,10188.2000,2.105000e+05
2,FC-A-3-11242,2025-01-03,NaN,MARRON OSCURO 77909 (16361221),25,9156.750000,21.0,11079.6675,2.289188e+05
3,FC-A-3-11250,2025-01-07,NaN,BLANCO POLAR (16466355),75,7385.000000,21.0,8935.8500,5.538750e+05
4,FC-A-3-11249,2025-01-07,NaN,BLANCO POLAR (16466355),50,7385.000000,21.0,8935.8500,3.692500e+05
...,...,...,...,...,...,...,...,...,...
427,FC-A-3-12220,2025-12-29,NaN,AMARILLO XP 05-02152,20,15163.000000,NaN,NaN,3.032600e+05
428,FC-A-3-12220,2025-12-29,NaN,FLUOR NARANJA 204A 2990 1426184,25,14027.250000,NaN,NaN,3.506812e+05
429,FC-A-3-12222,2025-12-30,NaN,FLUOR ROSA 204A 8320,75,9735.000000,NaN,NaN,7.301250e+05
430,FC-A-3-12222,2025-12-30,NaN,FLUOR VERDE WH 04-05481,25,14602.500000,NaN,NaN,3.650625e+05


In [2]:
# 2. Filtrar Facturas (FC-) y Notas de Crédito (CC-)
df_comprobantes = df_weg[df_weg['CPTE.'].astype(str).str.startswith(('FC-', 'CC-'), na=False)].copy()

In [3]:
# 3. Formatear la fecha y crear la columna MES (YYYY-MM)
df_comprobantes['FECHA'] = pd.to_datetime(df_comprobantes['FECHA'])
df_comprobantes['MES'] = df_comprobantes['FECHA'].dt.to_period('M').astype(str)

In [4]:
# 4. Limpiar y asegurar columnas numéricas necesarias (CANT. y TOTAL)
cols_num = ['CANT.', 'TOTAL']
for col in cols_num:
    if df_comprobantes[col].dtype == object:
        df_comprobantes[col] = df_comprobantes[col].astype(str).str.replace(',', '.')
    df_comprobantes[col] = pd.to_numeric(df_comprobantes[col], errors='coerce').fillna(0)

In [5]:
# 5. Estandarizar la columna de gasto neto
# El TOTAL en el excel original ya representa el monto neto operado
df_comprobantes['Gasto_Neto'] = df_comprobantes['TOTAL']

In [6]:
# 6. Clasificar productos
def clasificar_producto(desc):
    desc = str(desc).upper()
    if 'FOSFODESENGRASE' in desc or 'CHEMISA' in desc:
        return 'QUIM'
    elif 'ROD HOLDER' in desc or 'DEFLECTOR' in desc or 'EQUIPO DE PINTURA' in desc:
        return 'EQUIPOS'
    else:
        return 'PNTR'

df_comprobantes['Categoria'] = df_comprobantes['DESCRIPCIÓN'].apply(clasificar_producto)

In [7]:
# 7. Agrupar por MES y Categoría (Las CC- restan automáticamente al tener valores negativos)
resumen = df_comprobantes.groupby(['MES', 'Categoria']).agg({
    'CANT.': 'sum',
    'Gasto_Neto': 'sum'
}).reset_index()

In [8]:
# 8. Pivotear la tabla
df_pivot = resumen.pivot(index='MES', columns='Categoria').fillna(0)
# Aplanar el multi-índice
df_pivot.columns = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

In [9]:
# 9. Estructurar el DataFrame Final con los nombres solicitados
df_final = pd.DataFrame()
df_final['MES'] = df_pivot['MES']

# PINTURAS
df_final['CANT_PINTURA'] = df_pivot.get('CANT._PNTR', 0)
df_final['TOTAL_PINTURA'] = df_pivot.get('Gasto_Neto_PNTR', 0)

# QUÍMICOS
df_final['CANT_QUIMICOS'] = df_pivot.get('CANT._QUIM', 0)
df_final['TOTAL_QUIMICOS'] = df_pivot.get('Gasto_Neto_QUIM', 0)

# EQUIPOS
df_final['GASTOS_EN_EQUIPOS'] = df_pivot.get('Gasto_Neto_EQUIPOS', 0)

# Redondeo final a 2 decimales
cols_a_redondear = [col for col in df_final.columns if col != 'MES']
df_final[cols_a_redondear] = df_final[cols_a_redondear].round(2)

print("--- DataFrame Final de Consumos Netos Mensuales ---")
display(df_final)

--- DataFrame Final de Consumos Netos Mensuales ---


,MES,CANT_PINTURA,TOTAL_PINTURA,CANT_QUIMICOS,TOTAL_QUIMICOS,GASTOS_EN_EQUIPOS
0,2025-01,1020.0,8291168.70,110.0,1002788.65,0.0
1,2025-02,775.0,6326574.00,90.0,792543.40,0.0
2,2025-03,955.0,8334031.80,130.0,1246422.70,191180.0
3,2025-04,2208.0,18382842.25,190.0,2022662.20,114787.4
4,2025-05,2500.0,20985638.50,220.0,2182612.50,0.0
5,2025-06,1810.0,14973642.50,210.0,2353136.00,13200000.0
6,2025-07,1525.0,14110217.00,565.0,5056436.25,0.0
7,2025-08,975.0,10008738.75,370.0,3485167.25,0.0
8,2025-09,1225.0,12510823.50,496.0,5508634.50,209780.0
9,2025-10,1695.0,17369956.25,345.0,4023382.50,0.0


In [10]:
df_final.to_excel('df_weg.xlsx', index=False)
print('DataFrame saved to df_weg.xlsx')

DataFrame saved to df_weg.xlsx
